# Credit Card Fraud Detection

**Objective:**
Detect fraudulent credit card transactions in a severely imbalanced dataset.

**Dataset:**
Taken from Kaggle: <a href = "https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
">https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud</a>
- 284,807 transactions, 492 fraud cases (0.17% positive)
- Features: V1–V28 (PCA-transformed for anonymity) + Time + Amount
- Target: Class (0 = legitimate, 1 = fraud)

### Importing Dependencies

In [26]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR

import os
import json
import pickle

### Setting Configs

In [27]:
def set_seed(seed = 42):
    torch.manual_seed(seed)
    np.random.seed(seed)

set_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

### Loading Dataset

In [28]:
df = pd.read_csv('creditcarddata/creditcard.csv')
df.shape

(284807, 31)

In [29]:
def loading_dataset():
    data_path = Path("creditcard.csv")
    if data_path.exists():
        df = pd.read_csv(data_path)
        return df

    np.random.seed(42)
    N_LEGIT = 28000
    N_FRAUD = 49     # preserving 0.176% ratio

    # Generating PCA like features
    V_LEGIT = np.random.randn(N_LEGIT, 28)
    V_FRAUD = np.random.randn(N_FRAUD, 28) * 1.5 + np.random.randn(1, 28) * 2
    
    TIME_LEGIT = np.sort(np.random.uniform(0, 172800, N_LEGIT))
    TIME_FRAUD = np.random.uniform(0, 172800, N_FRAUD)
    
    AMT_LEGIT = np.random.exponential(88, N_LEGIT) 
    AMT_FRAUD = np.random.exponential(88, N_FRAUD)
    
    col_names = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount', 'Class']

    LEGIT_DATA = np.column_stack([TIME_LEGIT, V_LEGIT, AMT_LEGIT, np.zeros(N_LEGIT)])
    FRAUD_DATA = np.column_stack([TIME_FRAUD, V_FRAUD, AMT_FRAUD, np.ones(N_FRAUD)])

    df = pd.DataFrame(np.vstack([LEGIT_DATA, FRAUD_DATA]), columns = col_names)
    df['Class'] = df['Class'].astype(int)
    df = df.sample(frac = 1, random_state = 42).reset_index(drop = True)
    print(f'Dataset: {df.shape}')
    
    return df

df = loading_dataset()
print(f'Total Transactions: {len(df)}')
print(f'Fraud Cases: {df['Class'].sum():,}')
print(f'Fraud Rate: {df['Class'].mean():.4%}')
print(f'Features: {df.shape[1] - 1}')

Dataset: (28049, 31)
Total Transactions: 28049
Fraud Cases: 49
Fraud Rate: 0.1747%
Features: 30


### Preprocessing

In [30]:
feature_cols = [i for i in df.columns if i != 'Class']
X = df[feature_cols].values.astype(np.float32)
y = df['Class'].values.astype(np.int64)

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size = 0.2, 
                                                  random_state = 42, stratify = y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size = 0.176, 
                                                  random_state = 42, stratify = y_temp)

print(f'Train: {len(X_train):>8,}, {y_train.sum():>4} fraud, {y_train.mean():.4%}')
print(f'Val: {len(X_val):>8,}, {y_val.sum():>4} fraud, {y_val.mean():.4%}')
print(f'Test: {len(X_test):>8,}, {y_test.sum():>4} fraud, {y_test.mean():.4%}')

Train:   18,489,   32 fraud, 0.1731%
Val:    3,950,    7 fraud, 0.1772%
Test:    5,610,   10 fraud, 0.1783%


In [31]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [32]:
sm = SMOTE(random_state = 42, k_neighbors = 5)
X_train_resampled, y_train_resampled = sm.fit_resample(X_train_scaled, y_train)

print(f'Before SMOTE: {len(X_train_scaled):,} samples, ({y_train.sum():} fraud)')
print(f'After SMOTE: {len(X_train_resampled):,} samples, ({y_train_resampled.sum():} fraud),' 
f'{y_train_resampled.mean():1%} rate')

Before SMOTE: 18,489 samples, (32 fraud)
After SMOTE: 36,914 samples, (18457 fraud),50.000000% rate


### DataSet and DataLoader

In [33]:
class FraudDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype = torch.float32)
        self.y = torch.tensor(y, dtype = torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = FraudDataset(X_train_resampled, y_train_resampled)
val_dataset = FraudDataset(X_val_scaled, y_val)
test_dataset = FraudDataset(X_test_scaled, y_test)

train_loader = DataLoader(train_dataset, batch_size = 256, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 512, shuffle = False)
test_loader = DataLoader(test_dataset, batch_size = 512, shuffle = False)

print(f'Train Batches / Epochs : {len(train_loader)}')
print(f'Val Batches : {len(val_loader)}')

Train Batches / Epochs : 145
Val Batches : 8


### Focal Loss

In [34]:
class FocalLoss(nn.Module):
    def __init__(self, alpha:float = 0.25, gamma:float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits:torch.Tensor, targets:torch.Tensor) -> torch.Tensor:
        probs = torch.sigmoid(logits)
        probs_true = probs * targets + (1 - probs) * (1 - targets)
        
        alpha = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        weight = alpha * (1 - probs_true) ** self.gamma

        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction = 'none'
        )
        loss = weight * bce
        
        return loss.mean()

### Residual MLP Model

In [35]:
class ResModel(nn.Module):
    def __init__(self, dim:int, dropout:float = 2.0):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim)
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        return self.relu(self.model(x) + x)

class FraudDetectionNet(nn.Module):
    def __init__(self, in_features:int, hidden_dim:int = 256, n_blocks:int = 4, dropout:float = 0.2):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU()
        )
        self.model = nn.ModuleList([
            ResModel(hidden_dim, dropout) for _ in range(n_blocks) 
        ])
        self.output = nn.Linear(hidden_dim, 1)

        for i in self.modules():
            if isinstance(i, nn.Linear):
                nn.init.kaiming_normal_(i.weight, nonlinearity = 'relu')
                nn.init.zeros_(i.bias)

    def forward(self, x):
        x = self.input_proj(x)
        for block in self.model:
            x = block(x)
        return self.output(x)


model = FraudDetectionNet(
    in_features = len(feature_cols),
    hidden_dim = 256,
    n_blocks = 4,
    dropout = 0.2
).to(DEVICE)

print(f'Model Params: {sum(i.numel() for i in model.parameters()):,}')

Model Params: 539,137


### Training

In [38]:
criterion = FocalLoss(alpha = 0.25, gamma = 2.0)
optimizer = optim.AdamW(model.parameters(), lr = 1e-3, weight_decay = 1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max = 40, eta_min = 1e-6)

def precision_auc(y_true, y_prob, n_thresh = 100):
    thresholds = np.linspace(0, 1, n_thresh)
    precisions, recalls = [], []
    eps = 1e-8

    for i in thresholds:
        pred = (y_prob >= i).astype(int)
        tp = ((pred == 1) & (y_true == 1)).sum()
        fp = ((pred == 1) & (y_true == 0)).sum()
        fn = ((pred == 0) & (y_true == 1)).sum()

        precisions.append(tp / (tp + fp + eps))
        recalls.append(tp / (tp + fn + eps))

    idx = np.argsort(recalls)
    precisions = np.array(precisions)[idx]
    recalls = np.array(recalls)[idx]
    
    return float(np.trapz(precisions, recalls))

def run_epoch_fraud(model, loader, criterion, optimizer = None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_probs, all_labels = [], []

    ctx = torch.enable_grad() if is_training else torch.no_grad()
    with ctx:
        for bX, by in loader:
            bX, by = bX.to(DEVICE), by.to(DEVICE)

            if is_training:
                optimizer.zero_grad()
            logits = model(bX)
            loss = criterion(logits, by)
            
            if is_training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * bX.size(0)
            all_probs.append(torch.sigmoid(logits).detach().cpu().numpy().squeeze())
            all_labels.append(by.cpu().numpy().squeeze())

    probs = np.concatenate(all_probs)
    labels = np.concatenate(all_labels)
    return total_loss / len(loader.dataset), probs, labels

N_EPOCHS = 40
best_val_prauc = 0.0
history = []

os.makedirs('project1_fraud', exist_ok = True)

for epoch in range(N_EPOCHS):
    train_loss, _, _ = run_epoch_fraud(model, train_loader, criterion, optimizer)
    val_loss, v_probs, v_lbls = run_epoch_fraud(model, val_loader, criterion)

    val_prauc = precision_auc(v_lbls, v_probs)
    scheduler.step()

    history.append(f'Epoch: {epoch + 1}, Train_Loss: {train_loss}, Val_Loss: {val_loss}, val_prauc: {val_prauc}')

    if val_prauc > best_val_prauc:
        best_val_prauc = val_prauc
        torch.save(model.state_dict(), 'project1_fraud/best_model.pth')

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1:>3} | "
              f"Loss: {train_loss:.4f} / {val_loss:.4f} | Val PR-AUC: {val_prauc:.4f}")
print('TRAINING COMPLETED')

C:\Users\bisht\AppData\Local\Temp\ipykernel_25432\2572477034.py:23: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(precisions, recalls))


Epoch   5 | Loss: 0.0000 / 0.0000 | Val PR-AUC: 0.0716
Epoch  10 | Loss: 0.0000 / 0.0000 | Val PR-AUC: 0.0716
Epoch  15 | Loss: 0.0000 / 0.0000 | Val PR-AUC: 0.0716
Epoch  20 | Loss: 0.0000 / 0.0000 | Val PR-AUC: 0.0716
Epoch  25 | Loss: 0.0000 / 0.0000 | Val PR-AUC: 0.0716
Epoch  30 | Loss: 0.0000 / 0.0000 | Val PR-AUC: 0.0716
Epoch  35 | Loss: 0.0000 / 0.0000 | Val PR-AUC: 0.0716
Epoch  40 | Loss: 0.0000 / 0.0000 | Val PR-AUC: 0.0716
TRAINING COMPLETED


### Threshold Analysis

In [39]:
model.load_state_dict(torch.load('project1_fraud/best_model.pth', map_location = DEVICE))
_, v_probs, v_lbls = run_epoch_fraud(model, val_loader, criterion)

print('Threshold Analysis')
print(f"{'Thresh':>7} {'Prec':>7} {'Recall':>8} {'F1':>7} {'FN':>6} {'FP':>6}")

eps = 1e-8
for i in np.arange(0.1, 0.9, 0.1):
    pred = (v_probs >= i).astype(int)
    tp = int(((pred == 1) & (v_lbls == 1)).sum())
    fp = int(((pred == 1) & (v_lbls == 0)).sum())
    fn = int(((pred == 0) & (v_lbls == 1)).sum())
    prec = tp / (tp + fp + eps)
    rec = tp / (tp + fn + eps)
    f1 = 2 * prec * rec / (prec + rec + eps)

    print(f'{tp:>7.1f} {prec:>7.4f} {rec:>8.4f} {f1:>7.4f} {fn:>6} {fp:>6}')


print("So here, if missing fraud is worse than blocking customers -> we use lower threshold")
print("If customer friction matters more -> we use higher threshold")

Threshold Analysis
 Thresh    Prec   Recall      F1     FN     FP
    6.0  1.0000   0.8571  0.9231      1      0
    6.0  1.0000   0.8571  0.9231      1      0
    6.0  1.0000   0.8571  0.9231      1      0
    6.0  1.0000   0.8571  0.9231      1      0
    6.0  1.0000   0.8571  0.9231      1      0
    6.0  1.0000   0.8571  0.9231      1      0
    6.0  1.0000   0.8571  0.9231      1      0
    6.0  1.0000   0.8571  0.9231      1      0
So here, if missing fraud is worse than blocking customers -> we use lower threshold
If customer friction matters more -> we use higher threshold


### Test Set Evaluation

In [40]:
_, test_probs, test_lbls = run_epoch_fraud(model, test_loader, criterion)
test_prauc = precision_auc(test_lbls, test_probs)

# Using threshold=0.3 (recall-optimized for fraud detection)
t = 0.3
pred = (test_probs >= t).astype(int)
tp = int(((pred == 1) & (test_lbls == 1)).sum())
fp = int(((pred == 1) & (test_lbls == 0)).sum())
fn = int(((pred == 0) & (test_lbls == 1)).sum())
tn = int(((pred == 0) & (test_lbls == 0)).sum())
prec = tp / (tp + fp + eps)
rec  = tp / (tp + fn + eps)
f1   = 2 * prec * rec / (prec + rec + eps)

print(f"  PR-AUC:    {test_prauc:.4f}")
print(f"  Precision: {prec:.4f}  ({fp} false alarms)")
print(f"  Recall:    {rec:.4f}  ({fn} fraud cases MISSED)")
print(f"  F1:        {f1:.4f}")
print(f"\n  Confusion Matrix:")
print(f"               Pred Legit  Pred Fraud")
print(f"  True Legit:     {tn:>7}      {fp:>7}")
print(f"  True Fraud:     {fn:>7}      {tp:>7}")

  PR-AUC:    0.3506
  Precision: 1.0000  (0 false alarms)
  Recall:    1.0000  (0 fraud cases MISSED)
  F1:        1.0000

  Confusion Matrix:
               Pred Legit  Pred Fraud
  True Legit:        5600            0
  True Fraud:           0           10


C:\Users\bisht\AppData\Local\Temp\ipykernel_25432\2572477034.py:23: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return float(np.trapz(precisions, recalls))
